In [1]:
!pip install --upgrade transformers

  Using cached transformers-5.6.2-py3-none-any.whl.metadata (33 kB)
Using cached transformers-5.6.2-py3-none-any.whl (10.4 MB)
  Attempting uninstall: transformers
    Found existing installation: transformers 5.5.0
    Uninstalling transformers-5.5.0:
      Successfully uninstalled transformers-5.5.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth 2026.4.8 requires transformers!=4.52.0,!=4.52.1,!=4.52.2,!=4.52.3,!=4.53.0,!=4.54.0,!=4.55.0,!=4.55.1,!=4.57.0,!=4.57.4,!=4.57.5,!=5.0.0,!=5.1.0,<=5.5.0,>=4.51.3, but you have transformers 5.6.2 which is incompatible.
unsloth-zoo 2026.4.9 requires transformers!=4.52.0,!=4.52.1,!=4.52.2,!=4.52.3,!=4.53.0,!=4.54.0,!=4.55.0,!=4.55.1,!=4.57.4,!=4.57.5,!=5.0.0,!=5.1.0,<=5.5.0,>=4.51.3, but you have transformers 5.6.2 which is incompatible.


In [2]:
!pip install unsloth trl

  Using cached transformers-5.5.0-py3-none-any.whl.metadata (32 kB)
Using cached transformers-5.5.0-py3-none-any.whl (10.2 MB)
  Attempting uninstall: transformers
    Found existing installation: transformers 5.6.2
    Uninstalling transformers-5.6.2:
      Successfully uninstalled transformers-5.6.2


In [3]:
from unsloth import FastLanguageModel
from datasets import load_dataset, Dataset
from trl import SFTTrainer, SFTConfig
import pandas as pd
import json
import torch
import re
import ast
from transformers import TrainerCallback
import torch
from transformers import AutoTokenizer

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [4]:
import warnings

# Ignore all warnings
warnings.filterwarnings('ignore')

In [5]:
# ==========================================================
# Qwen2.5-7B-Instruct QLoRA fine-tuning for JSON macro extraction
# GPU: A100 80GB
# Library: unsloth
# ==========================================================
## СЛАВА КОЛАБУ ПРО

In [6]:
max_seq_length = 4096
load_in_4bit = False  ## У НАС МНОГО ПАМЯТИ
dtype = torch.bfloat16

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-7B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

unsloth/Qwen2.5-7B-Instruct does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [7]:
## LORA
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = 64, ## 2r
    lora_dropout = 0.1, ## делаем побольше для того, чтобы модель не переобучалась
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.1.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.4.8 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


In [8]:
## БЕРЕМ ВЕСЬ ДАТАСЕТ
data_2000 = pd.read_csv("df_2000_all_new.csv")
data_2000

,процентная ставка,ВВП,инфляция,безработица,капитал,инвестиции,производство,потребление,численность рабочей силы,сбережения,...,доходы населения,валютный курс,импорт,экспорт,государственные расходы,государственный долг,дефицит бюджета,doc_id,chunk_id,original_text
0,not stated,not stated,not stated,not stated,down,not stated,not stated,not stated,not stated,not stated,...,not stated,not stated,not stated,not stated,not stated,not stated,not stated,1,0,"""Индекс Мосбиржи -3%. Это самое большое дневно..."
1,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,...,not stated,not stated,not stated,not stated,not stated,not stated,not stated,2,0,"""​​Циан взлетел на апелляции Крупнейший в Росс..."
2,not stated,down,up,not stated,not stated,not stated,down,not stated,not stated,not stated,...,down,not stated,not stated,down,not stated,not stated,up,2,1,"""С 5 февраля ЕС вводит эмбарго на российские п..."
3,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,...,not stated,not stated,not stated,not stated,not stated,not stated,up,3,0,Госбюджет РФ рискует недополучить еще больше н...
4,not stated,not stated,not stated,not stated,up,not stated,not stated,not stated,not stated,not stated,...,not stated,not stated,not stated,not stated,not stated,not stated,not stated,4,0,"""Новый отчёт Сбера (SBER): чистая прибыль за 1..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1801,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,...,not stated,not stated,not stated,not stated,not stated,not stated,not stated,1488,0,• Объем торгов металлами в феврале составил 61...
1802,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,...,not stated,not stated,not stated,not stated,not stated,not stated,not stated,1489,0,"""​​Страхованию инвестиций на ИИС — быть В дека..."
1803,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,...,not stated,up,not stated,not stated,not stated,not stated,not stated,1489,1,"""​​Что означает 100 рублей за доллар для префо..."
1804,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,not stated,...,not stated,up,not stated,not stated,not stated,not stated,not stated,1491,0,В начале октября валютный курс тестирует сопро...


In [9]:
## КОСЯКА ЗДЕСЬ ТОЧНО НЕ БЫЛО)
data_2000 = data_2000[data_2000['doc_id'] != 'up']
data_2000['doc_id'] = data_2000['doc_id'].astype('int')

In [10]:
## ИДЕАЛЬНОЕ РАЗБИЕНИЕ
test = data_2000[(data_2000['doc_id'] > 1234)]
val = data_2000[(data_2000['doc_id'] > 1000) & (data_2000['doc_id'] <= 1234)]
train = data_2000[data_2000['doc_id'] <= 1000]

In [11]:
train.shape, val.shape, test.shape

((1243, 21), (292, 21), (268, 21))

In [12]:
## ДЕЛАЕМ ДАТЕСЕТЫ ДЛЯ УДОБСТВА И ПЕРЕКЛАДЫВАЕМ В JSON
train_new = [{"json_target": str(dict(train.iloc[i, :18])), "text": train.iloc[0, -1]} for i in range(len(train))]
train_new = pd.DataFrame(train_new)

val_new = [{"json_target": str(dict(val.iloc[i, :18])), "text": val.iloc[0, -1]} for i in range(len(val))]
val_new = pd.DataFrame(val_new)

test_new = [{"json_target": str(dict(val.iloc[i, :18])), "text": val.iloc[0, -1]} for i in range(len(val))]
test_new = pd.DataFrame(val_new)

In [13]:
test_new

,json_target,text
0,"{'процентная ставка': 'not stated', 'ВВП': 'no...",Эти действия существенно снизят будущие риски ...
1,"{'процентная ставка': 'not stated', 'ВВП': 'no...",Эти действия существенно снизят будущие риски ...
2,"{'процентная ставка': 'up', 'ВВП': 'not stated...",Эти действия существенно снизят будущие риски ...
3,"{'процентная ставка': 'up', 'ВВП': 'not stated...",Эти действия существенно снизят будущие риски ...
4,"{'процентная ставка': 'not stated', 'ВВП': 'no...",Эти действия существенно снизят будущие риски ...
...,...,...
287,"{'процентная ставка': 'up', 'ВВП': 'down', 'ин...",Эти действия существенно снизят будущие риски ...
288,"{'процентная ставка': 'not stated', 'ВВП': 'no...",Эти действия существенно снизят будущие риски ...
289,"{'процентная ставка': 'up', 'ВВП': 'not stated...",Эти действия существенно снизят будущие риски ...
290,"{'процентная ставка': 'up', 'ВВП': 'not stated...",Эти действия существенно снизят будущие риски ...


In [14]:
## ЕЩЕ ДВА КРУТЫХ ДАТАСЕТА
train_df = train_new
val_df = val_new

In [15]:
### ДЕЛАЕМ ДАТАСЕТ ДЛЯ ОБУЧЕНИЯ
def prepare_train_dataset(df, tokenizer):
    """Функция для полной подготовки датасета к обучению"""

    def formatting_prompts_func(examples):
        instructions = []
        for text, target in zip(examples["text"], examples["json_target"]):

            user_prompt = (
                "Твоя задача — анализировать новости "
                "и определять динамику 18 показателей. Оценивай прямые упоминания и логические следствия. "
                "Допустимые значения изменений ТОЛЬКО ТАКИЕ: 'up', 'down', 'not stated'. "
                "Ответь строго в формате JSON без вступлений и пояснений: "
                "{"
                "\"процентная ставка\": \"тип изменения\","
                "\"ВВП\": \"тип изменения\","
                "\"инфляция\": \"тип изменения\","
                "\"безработица\": \"тип изменения\","
                "\"капитал\": \"тип изменения\","
                "\"инвестиции\": \"тип изменения\","
                "\"производство\": \"тип изменения\","
                "\"потребление\": \"тип изменения\","
                "\"численность рабочей силы\": \"тип изменения\","
                "\"сбережения\": \"тип изменения\","
                "\"заработные платы\": \"тип изменения\","
                "\"доходы населения\": \"тип изменения\","
                "\"валютный курс\": \"тип изменения\","
                "\"импорт\": \"тип изменения\","
                "\"экспорт\": \"тип изменения\","
                "\"государственные расходы\": \"тип изменения\","
                "\"государственный долг\": \"тип изменения\","
                "\"дефицит бюджета\": \"тип изменения\""
                "} "
                f"Текст новости: {text}"
            )

            messages = [
                {"role": "system", "content": "Ты — экспертный макроэкономический аналитик."},
                {"role": "user", "content": user_prompt}
            ]

            # Применяем CHAT TEMPLATE
            formatted_text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )

            if isinstance(target, dict):
                target_str = json.dumps(target, ensure_ascii=False)
            else:
                target_str = str(target).strip()

            # СКЛЕИВАЕМ Промпт + Ответ + EOS токен
            full_text = f"{formatted_text}{target_str}{tokenizer.eos_token}"
            instructions.append(full_text)

        return {"text": instructions}

    # HuggingFace Dataset
    dataset = Dataset.from_pandas(df[['text', 'json_target']])

    dataset = dataset.map(formatting_prompts_func, batched=True)

    return dataset

In [27]:
## ДЕЛАЕМ ДАТАСЕТЫ ФОРМАТА CLEAR ML
train_dataset = prepare_train_dataset(train_df, tokenizer)
val_dataset = prepare_train_dataset(val_df, tokenizer)
test_dataset = prepare_train_dataset(test_new, tokenizer)


Map:   0%|          | 0/1243 [00:00<?, ? examples/s]

Map:   0%|          | 0/292 [00:00<?, ? examples/s]

Map:   0%|          | 0/292 [00:00<?, ? examples/s]

In [19]:
# Настраиваем тренер
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = val_dataset,
    dataset_text_field = "text",
    max_seq_length = 4096,
    packing = False,  # ДЛЯ РАБОТЫ С JSON-ОМ НЕ НАДО
    args = SFTConfig(
        per_device_train_batch_size = 48,
        gradient_accumulation_steps = 1,
        num_train_epochs = 5, ## БЕРЕМ ПОБОЛЬШЕ ЭПОХ, Т.К В НЕКОТОРЫХ КАТЕОГРИЯХ МАЛО ДАННЫХ ПЛЮС ЕСТЬ СОХРАНЕНИЕ ЛУЧШЕЙ МОДЕЛИ
        learning_rate = 1e-5,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.1,
        output_dir = "outputs",
        save_strategy = "epoch",
        eval_strategy = "epoch",
        metric_for_best_model = "eval_loss",
        load_best_model_at_end = True,
        save_total_limit = 3,
        report_to = "none",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/1243 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/292 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [20]:
# СТАРТУЕМ
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,243 | Num Epochs = 5 | Total steps = 130
O^O/ \_/ \    Batch size per device = 48 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (48 x 1 x 1) = 48
 "-____-"     Trainable parameters = 80,740,352 of 7,696,356,864 (1.05% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!


Epoch,Training Loss,Validation Loss
1,1.241530,1.036581
2,0.482958,0.770179
3,0.154323,0.747203
4,0.056112,0.750823
5,0.047077,0.747348


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-26/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-52/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-78/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-104/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-130/tokenizer_config.json.


TrainOutput(global_step=130, training_loss=0.43223280081382165, metrics={'train_runtime': 1079.316, 'train_samples_per_second': 5.758, 'train_steps_per_second': 0.12, 'total_flos': 1.4297791689312768e+17, 'train_loss': 0.43223280081382165, 'epoch': 5.0})

In [21]:
### СОХРАНИМ
model.save_pretrained("qwen_macro_lora")
tokenizer.save_pretrained("qwen_macro_lora")

Unsloth: Restored added_tokens_decoder metadata in qwen_macro_lora/tokenizer_config.json.


('qwen_macro_lora/tokenizer_config.json',
 'qwen_macro_lora/chat_template.jinja',
 'qwen_macro_lora/tokenizer.json')

In [22]:
## сначала в zip
!zip -r qwen_macro_lora.zip ./qwen_macro_lora

  adding: qwen_macro_lora/ (stored 0%)
  adding: qwen_macro_lora/README.md (deflated 65%)
  adding: qwen_macro_lora/tokenizer.json (deflated 81%)
  adding: qwen_macro_lora/chat_template.jinja (deflated 71%)
  adding: qwen_macro_lora/tokenizer_config.json (deflated 89%)
  adding: qwen_macro_lora/adapter_model.safetensors (deflated 8%)
  adding: qwen_macro_lora/adapter_config.json (deflated 57%)


In [23]:
## сохраним в облако
!curl -F "file=@./qwen_macro_lora.zip" https://store1.gofile.io/uploadFile

{"data":{"createTime":1777204778,"downloadPage":"https://gofile.io/d/dtahND","guestToken":"hss8hShsDp3zOI4dko5KJpMIacAznCdw","id":"8140bb63-176e-420b-8bed-172c483f3d4f","md5":"976ac8a22591574d5057e480ee354405","mimetype":"application/zip","modTime":1777204778,"name":"qwen_macro_lora.zip","parentFolder":"d4050f2c-dc20-4b8a-9c39-f82d60869ace","parentFolderCode":"dtahND","servers":["store1"],"size":299304948,"type":"file"},"status":"ok"}


## ТЕСТ НОВОЙ МОДЕЛИ

In [55]:
## ЕЩЕ РАЗ ДЕЛАЕМ ТЕСТОВВЫЙ ДАТАСЕТ
test_data = pd.DataFrame(test_dataset)
test_dataset = list(test['original_text'])

In [65]:
responces = []
with torch.no_grad():
    for i, text in enumerate(test_dataset, 1):
        print(f'Обработка документа {i}...')


        prompt = (test_data.iloc[i, 0])
        messages = [
            {"role": "system", "content": "Ты — экспертный макроэкономический аналитик."},
            {"role": "user", "content": prompt}
        ]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=512
        )
        generated_ids = [
            output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]
        response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
        responces.append(response)

Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Обработка документа 1...


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Обработка документа 2...


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Обработка документа 266...


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Обработка документа 267...


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Обработка документа 268...


In [66]:
df_to_check = test_data

## СЧИТАЕМ МЕТРИКИ ДЛЯ ДООБУЧЕННОЙ МОДЕЛИ НА ТЕСТЕ

In [67]:
expected_keys = [
    "процентная ставка", "ввп", "инфляция", "безработица", "капитал",
    "инвестиции", "производство", "потребление", "численность рабочей силы",
    "сбережения", "заработные платы", "доходы населения", "валютный курс",
    "импорт", "экспорт", "государственные расходы", "государственный долг",
    "дефицит бюджета"
]

total_accuracy = 0
valid_json_count = 0
up_down_accuracy = 0
not_up_down_total = 0

for i, pred_raw in enumerate(responces):
    try:
        clean_pred = pred_raw.strip()

        json_match = re.search(r'\{.*\}', clean_pred, re.DOTALL)
        if not json_match:
            print(f"Документ {i+1}: JSON не найден")
            continue

        json_str = json_match.group()

        try:
            raw_pred_json = json.loads(json_str)
        except json.JSONDecodeError:
            raw_pred_json = ast.literal_eval(json_str)

        pred_json = {
            str(k).lower().replace('капital', 'капитал').replace('kapital', 'капитал').strip(): v
            for k, v in raw_pred_json.items()
        }

        row = df_to_check.iloc[i]
        target_raw = row['json_target']

        if isinstance(target_raw, str):
            try:
                target_dict = json.loads(target_raw)
            except:
                target_dict = ast.literal_eval(target_raw)
        else:
            target_dict = target_raw

        target_dict = {str(k).lower().strip(): v for k, v in target_dict.items()}

        correct_in_doc = 0
        correct_up_down = 0
        count_up_down = 0

        for entity in expected_keys:
            model_val = str(pred_json.get(entity, "not stated")).lower().strip()
            true_val = str(target_dict.get(entity, "not stated")).lower().strip()

            if model_val == true_val:
                correct_in_doc += 1

            if true_val != 'not stated':
                count_up_down += 1
                if model_val == true_val:
                    correct_up_down += 1

        if count_up_down > 0:
            up_down_accuracy += (correct_up_down / count_up_down)
        else:
            not_up_down_total += 1

        total_accuracy += (correct_in_doc / len(expected_keys))
        valid_json_count += 1

    except Exception as e:
        print(f"Ошибка в документе {i+1}: {type(e).__name__} - {e}")

# Итоговые цифры
k = len(responces)
print("\n" + "="*30)
print(f"ОБРАБОТАНО УСПЕШНО: {valid_json_count} из {k}")
if k > 0:
    print(f"ОБЩАЯ ACCURACY (по всем ключам): {total_accuracy/k:.4f}")
    divisor = (k - not_up_down_total)
    if divisor > 0:
        print(f"ACCURACY (только UP/DOWN): {up_down_accuracy/divisor:.4f}")

Ошибка в документе 55: SyntaxError - invalid syntax (<unknown>, line 3)
Ошибка в документе 84: SyntaxError - invalid character '，' (U+FF0C) (<unknown>, line 6)
Ошибка в документе 214: SyntaxError - invalid syntax (<unknown>, line 3)

ОБРАБОТАНО УСПЕШНО: 265 из 268
ОБЩАЯ ACCURACY (по всем ключам): 0.7937
ACCURACY (только UP/DOWN): 0.2580


## ИСПОЛЬЗОВАНИЕ РАСПАКОВАННОЙ МОДЕЛИ

In [ ]:
# Распаковываем
!unzip qwen_macro_lora.zip

In [ ]:
from peft import PeftModel

model_id ="Qwen/Qwen2.5-7B-Instruct" # Исходная модель
adapter_path = "qwen_macro_lora" # Ваша папка

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

model = PeftModel.from_pretrained(base_model, adapter_path)

tokenizer = AutoTokenizer.from_pretrained(adapter_path)